# Semana 32 — OEC: a Métrica Certa e o Capstone Final
**Fase 7: Governance e Engineering at Scale** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/07-Phase-7-Governance-and-Scale/notebooks/semana_32_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-32-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/07-Phase-7-Governance-and-Scale/docs/Semana-32-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_32_gabarito.ipynb`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import precision_score, recall_score

## Passo 1 — O histórico de experimentos
240 experimentos passados de um produto com recomendação por IA. Para cada um medimos, durante o teste, o efeito (lift, em %) sobre quatro métricas de curto prazo; meses depois soubemos o efeito sobre a **retenção em 90 dias**, que é o que a empresa de fato quer.

Há três tipos de mudança: melhorias reais de relevância, mudanças de "isca de clique" e mudanças neutras.

In [ ]:
def historico(n=240, seed=0):
    r = np.random.default_rng(seed); tipo = r.choice(["relevancia", "isca", "neutro"], n, p=[0.35, 0.30, 0.35]); q = r.normal(0, 1, n)
    f = lambda rel, isca, neu: np.where(tipo == "relevancia", rel, np.where(tipo == "isca", isca, neu))
    cliques = f(1.5 + 0.8 * q, 4.0 + 1.0 * q, 0.2 * q) + r.normal(0, 0.8, n)
    sucesso = f(2.0 + 1.0 * q, -1.5 + 0.6 * q, 0.2 * q) + r.normal(0, 0.8, n)          # sessões em que o usuário achou o que queria
    retorno7 = f(1.2 + 0.7 * q, -0.8 + 0.5 * q, 0.1 * q) + r.normal(0, 0.7, n)         # voltou em 7 dias
    reclam = f(-0.5 + 0.3 * q, 2.5 + 0.8 * q, 0.1 * q) + r.normal(0, 0.6, n)           # reclamações
    retencao90 = f(1.0 + 0.6 * q, -1.2 + 0.5 * q, 0.05 * q) + r.normal(0, 0.25, n)     # o que importa
    return pd.DataFrame({"tipo": tipo, "cliques": cliques, "sucesso_sessao": sucesso, "retorno_7d": retorno7, "reclamacoes": reclam, "retencao_90d": retencao90})

exp = historico(); print(exp.groupby("tipo").mean().round(2)); exp.head()

## Passo 2 — A proxy enganosa
Hoje a empresa lança todo experimento com lift positivo em cliques. Calcule a correlação entre o lift de cliques e o de retenção e visualize.

In [ ]:
# [SEU CÓDIGO AQUI] correlação entre cliques e retencao_90d
print(f"correlação cliques × retenção 90d: {corr:.2f}")
for t, c in [("relevancia", "g"), ("isca", "crimson"), ("neutro", "gray")]:
    d = exp[exp.tipo == t]; plt.scatter(d.cliques, d.retencao_90d, s=14, c=c, label=t, alpha=.7)
plt.axhline(0, c="k", lw=.6); plt.axvline(0, c="k", lw=.6); plt.xlabel("lift em cliques (%)"); plt.ylabel("lift em retenção 90d (%)"); plt.legend(); plt.show()

O quadrante inferior direito são experimentos que a regra atual **lança** e que **reduzem** a retenção.

## Passo 3 — A qualidade das decisões por cliques
Trate "lançar" como classe positiva e "o experimento realmente melhora a retenção" como verdade. Calcule precision e recall da regra `cliques > 0`.

In [ ]:
bom = (exp["retencao_90d"] > 0).astype(int)
def qualidade(decisao):
    # [SEU CÓDIGO AQUI] devolva lançados, precisão, recall e número de prejudiciais lançados
dec_cliques = (exp["cliques"] > 0).astype(int); print(qualidade(dec_cliques))

## Passo 4 — Construindo o OEC
Separe o histórico em ajuste (70%) e validação (30%). Ajuste uma regressão linear que preveja o lift de retenção em 90 dias a partir das quatro métricas de curto prazo. Os coeficientes são os pesos do surrogate index.

In [ ]:
CURTO = ["cliques", "sucesso_sessao", "retorno_7d", "reclamacoes"]
rng = np.random.default_rng(1); idx = rng.permutation(len(exp)); corte = int(0.7 * len(exp))
ajuste, valid = exp.iloc[idx[:corte]], exp.iloc[idx[corte:]]
# [SEU CÓDIGO AQUI] ajuste a regressão e guarde os coeficientes em `pesos`
print(pesos.round(3)); print("intercepto:", round(modelo.intercept_, 3))

Leia os sinais: que peso recebeu o clique, depois de controlar pelas outras métricas? E as reclamações?

## Passo 5 — Out-of-sample validation
No validation set: o OEC acompanha a retenção? E, principalmente, acerta o **sinal**?

In [ ]:
# [SEU CÓDIGO AQUI] OEC na validação, correlação com a retenção e fração de acerto do sinal
acerto_cliques = np.mean((valid["cliques"] > 0) == (valid["retencao_90d"] > 0))
print(f"correlação com retenção | OEC: {corr_oec:.2f} | cliques: {valid['cliques'].corr(valid['retencao_90d']):.2f}")
print(f"acerto do sinal         | OEC: {acerto_sinal:.2f} | cliques: {acerto_cliques:.2f}")
plt.scatter(oec_valid, valid["retencao_90d"], s=14); plt.axhline(0, c="k", lw=.6); plt.axvline(0, c="k", lw=.6); plt.xlabel("OEC (lift previsto de retenção)"); plt.ylabel("retenção 90d real"); plt.show()

## Passo 6 — Um OEC que as pessoas entendem
Uma regressão com quatro casas decimais não vira cultura. Simplifique: arredonde os pesos para uma fórmula como `sucesso + retorno − reclamações` (pesos inteiros pequenos, clique fora) e verifique quanto se perde.

In [ ]:
def oec_simples(d):
    # [SEU CÓDIGO AQUI] uma combinação simples, com pesos inteiros, sem cliques
s = oec_simples(valid)
print(f"OEC simples | correlação: {np.corrcoef(s, valid['retencao_90d'])[0, 1]:.2f} | acerto do sinal: {np.mean((s > 0) == (valid['retencao_90d'] > 0)):.2f}")

## Passo 7 — Guardrail metric
Mesmo com OEC positivo, um experimento que aumenta as reclamações em mais de 1% não é lançado. Monte a decisão final: `OEC simples > 0` **e** `reclamacoes <= 1`.

In [ ]:
# [SEU CÓDIGO AQUI] decisão com OEC e guardrail, para todo o histórico
print("cliques:", qualidade(dec_cliques)); print("OEC + guardrail:", qualidade(dec_oec))

## Passo 8 — Quanto vale a troca de métrica
Some o lift de retenção de todos os experimentos que cada regra lançaria. Compare com o oráculo (lançar exatamente os que melhoram a retenção).

In [ ]:
def valor(decisao):
    # [SEU CÓDIGO AQUI] soma do lift de retenção dos experimentos lançados
v = {"decidir por cliques": valor(dec_cliques), "decidir por OEC + guardrail": valor(dec_oec), "oráculo": valor(bom)}
for k, x in v.items(): print(f"{k:28s} | retenção acumulada: {x:+7.1f} pontos")
plt.bar(list(v), list(v.values()), color=["crimson", "seagreen", "gray"]); plt.ylabel("lift acumulado de retenção (pontos)"); plt.xticks(rotation=10); plt.show()

## Passo 9 — O que acontece quando o time otimiza a métrica
Simule um ano em que o time, cobrado pela métrica, passa a propor mais experimentos do tipo que a move. Sob a regra de cliques, a proporção de "iscas" cresce; sob o OEC, cresce a de melhorias de relevância. Compare o valor acumulado.

In [ ]:
def ano(prop, regra, n=240, seed=5):
    r = np.random.default_rng(seed); e = historico(n, seed)
    e["tipo_alvo"] = r.choice(["relevancia", "isca", "neutro"], n, p=prop)
    banco = {t: historico(2000, seed + i + 1).query("tipo == @t").reset_index(drop=True) for i, t in enumerate(["relevancia", "isca", "neutro"])}
    e = pd.concat([banco[t].iloc[[i % len(banco[t])]] for i, t in enumerate(e["tipo_alvo"])], ignore_index=True)
    # [SEU CÓDIGO AQUI] aplique a regra de decisão e devolva o lift de retenção acumulado dos lançados

print(f"time otimizando cliques (60% iscas)     : {ano([0.15, 0.60, 0.25], 'cliques'):+.1f}")
print(f"time otimizando o OEC (60% relevância)  : {ano([0.60, 0.15, 0.25], 'oec'):+.1f}")

Esse é o argumento central para a diretoria: a métrica não só decide quais experimentos são lançados; ela decide **quais experimentos o time propõe**.

## Fechamento da trilha
Escreva o memorando pedido no entregável e, em seguida, leia a descrição do projeto final na leitura de conceito desta semana.